# qlib Native Research Notebook

这份 notebook 现在只负责两件事：
- 配置并调用统一的 `qlib_native` 研究 runner。
- 读取统一产物，展示研究价值、执行质量、风险和稳定性诊断。

训练、打分、原生回测、shared-signal validation、diagnostics 落盘，都由脚本化模块负责，不再在 notebook 内重复维护一条平行逻辑。

## 1. 目标与使用方式

建议的使用姿势：
- 日常研究时，把 `RUN_WORKFLOW = False`，直接复盘已有产物。
- 需要刷新 baseline / candidate 时，把 `RUN_WORKFLOW = True`，让 notebook 直接调用统一 runner。
- 所有主流程参数都集中在一个 `CONFIG` 字典里，避免 notebook 和脚本参数漂移。

## 2. 环境检查与工作流入口

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qlib_research.core.notebook_workflow import (
    load_native_workflow_artifacts,
    run_native_notebook_workflow,
)


def _safe_last_numeric(frame: pd.DataFrame, column: str):
    if frame.empty or column not in frame.columns:
        return None
    series = pd.to_numeric(frame[column], errors="coerce").dropna()
    if series.empty:
        return None
    return float(series.iloc[-1])


def _safe_max_numeric(frame: pd.DataFrame, column: str):
    if frame.empty or column not in frame.columns:
        return None
    series = pd.to_numeric(frame[column], errors="coerce").dropna()
    if series.empty:
        return None
    return float(series.max())


def _show_line(frame: pd.DataFrame, x: str, y, title: str, height: int = 360):
    if frame.empty:
        display(Markdown(f"_没有可展示的数据：{title}_"))
        return
    fig = px.line(frame, x=x, y=y, title=title)
    fig.update_layout(template="plotly_white", height=height)
    fig.show()


def _prepare_heatmap_frame(heatmap_frame: pd.DataFrame) -> pd.DataFrame:
    if heatmap_frame.empty:
        return heatmap_frame
    frame = heatmap_frame.copy()
    if "year" in frame.columns:
        frame["year"] = pd.to_numeric(frame["year"], errors="coerce")
        frame = frame.loc[frame["year"].notna()].copy()
        frame["year"] = frame["year"].astype(int).astype(str)
        frame = frame.set_index("year")
    if "Unnamed: 0" in frame.columns:
        frame = frame.set_index("Unnamed: 0")
    frame = frame.apply(pd.to_numeric, errors="coerce")
    return frame


def _render_return_heatmap(heatmap_frame: pd.DataFrame, title: str, x_label: str, y_label: str) -> None:
    normalized_frame = _prepare_heatmap_frame(heatmap_frame)
    if normalized_frame.empty:
        display(Markdown(f"_没有可展示的数据：{title}_"))
        return
    numeric_frame = normalized_frame * 100.0
    finite_values = numeric_frame.to_numpy().ravel()
    finite_values = finite_values[pd.notna(finite_values)]
    if len(finite_values):
        abs_values = pd.Series(finite_values).abs()
        robust_bound = float(abs_values.quantile(0.90)) if not abs_values.empty else 1.0
        robust_bound = max(robust_bound, 1.0)
        symmetric_bound = min(robust_bound, 30.0)
    else:
        symmetric_bound = 1.0
    fig = px.imshow(
        numeric_frame,
        text_auto=".1f",
        aspect="auto",
        color_continuous_scale=[
            [0.0, "#b2182b"],
            [0.5, "#f7f7f7"],
            [1.0, "#1a9850"],
        ],
        zmin=-symmetric_bound,
        zmax=symmetric_bound,
        origin="lower",
        title=title,
        labels={"x": x_label, "y": y_label, "color": "Net Return %"},
    )
    fig.update_layout(height=420)
    fig.show()


runtime_check = pd.DataFrame(
    [
        {"check": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
        {"check": "Notebook role", "value": "thin native workflow viewer"},
        {"check": "Runner helper", "value": "run_native_notebook_workflow"},
        {"check": "Artifact loader", "value": "load_native_workflow_artifacts"},
    ]
)
display(runtime_check)


,check,value
0,PROJECT_ROOT,/Volumes/Repository/Projects/TradingNexus/Valu...
1,Notebook role,thin native workflow viewer
2,Runner helper,run_native_notebook_workflow
3,Artifact loader,load_native_workflow_artifacts


## 3. 参数区

In [2]:
RUN_WORKFLOW = False
RECIPE_NAMES = ["baseline", "mae_4w", "binary_4w", "rank_blended", "huber_8w"]
FOCUS_RECIPE = "huber_8w"

CONFIG = {
    "universe_profile": "csi300",
    "panel_path": "artifacts/panels/csi300_weekly.csv",
    "execution_panel_path": None,
    "output_dir": "artifacts/native_workflow/csi300",
    "run_export": "auto_if_missing",
    "start_date": "2016-01-01",
    "end_date": None,
    "batch_size": 200,
    "topk": 10,
    "train_weeks": 260,
    "valid_weeks": 52,
    "eval_count": 52,
    "step_weeks": 1,
    "walk_forward_enabled": True,
    "walk_forward_eval_count": 0,
    "walk_forward_train_weeks": 260,
    "walk_forward_valid_weeks": 52,
    "walk_forward_step_weeks": 4,
    "benchmark_mode": "auto",
    "signal_objective": "huber_regression",
    "label_recipe": "blended_excess_4w_8w",
    "rebalance_interval_weeks": 1,
    "hold_buffer_rank": 15,
    "universe_exit_policy": "retain_quotes_for_existing_positions",
    "min_liquidity_filter": 0.0,
    "min_score_spread": 0.0,
    "industry_max_weight": 0.30,
    "diagnostics_enabled": True,
    "run_validation_comparison": True,
    "publish_model": False,
}

display(pd.DataFrame([CONFIG]).T.rename(columns={0: "value"}))
display(Markdown("`benchmark_mode` 默认按 `universe_profile` 自动映射；也可以手工设为 `000001.SH` 或 `000001.SH@上证指数`。"))
display(Markdown("`walk_forward_eval_count=0` 表示使用全部可评估周；大于 0 时只保留最近 N 个评估周。"))


,value
universe_profile,csi300
panel_path,artifacts/panels/csi300_weekly.csv
execution_panel_path,None
output_dir,artifacts/native_workflow/csi300
run_export,auto_if_missing
start_date,2016-01-01
end_date,None
batch_size,200
topk,10
train_weeks,260


`benchmark_mode` 默认按 `universe_profile` 自动映射；也可以手工设为 `000001.SH` 或 `000001.SH@上证指数`。

`walk_forward_eval_count=0` 表示使用全部可评估周；大于 0 时只保留最近 N 个评估周。

## 4. 运行或加载统一研究工作流

In [3]:
workflow_run = run_native_notebook_workflow(
    config_overrides=CONFIG,
    recipe_names=RECIPE_NAMES,
    run_workflow=RUN_WORKFLOW,
)

artifact_view = load_native_workflow_artifacts(
    workflow_run["output_dir"],
    recipe_names=RECIPE_NAMES,
)
workflow_summary = workflow_run["workflow_summary"]

display(Markdown(f"**输出目录**: `{workflow_run['output_dir']}`"))
display(pd.DataFrame([workflow_summary.get("config", CONFIG)]).T.rename(columns={0: "value"}))

if artifact_view["promotion_gate"].empty:
    display(Markdown("当前没有可展示的 promotion gate 输出；如果只运行了 `baseline`，或还没有刷新 workflow，这是正常的。"))
else:
    display(artifact_view["promotion_gate"])


**输出目录**: `/Volumes/Repository/Projects/TradingNexus/QlibResearch/artifacts/native_workflow/csi300`

,value
universe_profile,csi300
panel_path,artifacts/panels/csi300_weekly.csv
execution_panel_path,None
output_dir,artifacts/native_workflow/csi300
start_date,2016-01-01
end_date,None
batch_size,200
run_export,auto_if_missing
topk,10
train_weeks,260


,recipe,promotion_gate_passed,baseline_topk_unique_score_ratio,candidate_topk_unique_score_ratio,baseline_signal_turnover_proxy,candidate_walk_forward_net_total_return,baseline_walk_forward_net_total_return,candidate_walk_forward_drawdown,baseline_walk_forward_drawdown
0,mae_4w,False,0.959524,0.860938,0.548,0.308333,0.651681,-0.281078,-0.205025
1,binary_4w,False,0.959524,0.215234,0.548,0.196119,0.651681,-0.367342,-0.205025
2,rank_blended,False,0.959524,0.913492,0.548,0.396291,0.651681,-0.165301,-0.205025
3,huber_8w,True,0.959524,0.961508,0.548,0.678031,0.651681,-0.148861,-0.205025


## 5. 工作流总览

In [4]:
recipe_overview = artifact_view["recipe_overview"].copy()
if recipe_overview.empty:
    display(Markdown("当前输出目录下还没有 recipe 产物。把 `RUN_WORKFLOW` 改成 `True` 后重新运行上一格。"))
else:
    recipe_overview = recipe_overview.sort_values("recipe").reset_index(drop=True)
    walk_forward_limit = workflow_summary.get("config", {}).get("walk_forward_eval_count")
    if pd.notna(walk_forward_limit) and int(walk_forward_limit) > 0:
        display(Markdown(f"当前 `walk_forward_eval_count={int(walk_forward_limit)}`，只会保留最近 {int(walk_forward_limit)} 个可评估周，而不是全历史 walk-forward。"))
    key_metric_columns = [
        "recipe",
        "rolling_rank_ic_ir",
        "rolling_topk_mean_excess_return_4w",
        "rolling_eval_date_count",
        "rolling_eval_date_start",
        "rolling_eval_date_end",
        "rolling_net_total_return",
        "rolling_max_drawdown",
        "walk_forward_rank_ic_ir",
        "walk_forward_topk_mean_excess_return_4w",
        "walk_forward_eval_date_count",
        "walk_forward_eval_date_start",
        "walk_forward_eval_date_end",
        "walk_forward_net_total_return",
        "walk_forward_max_drawdown",
        "latest_score_dispersion",
        "latest_top10_unique_score_count",
        "latest_actual_hold_count",
        "latest_blocked_sell_count",
    ]
    available_columns = [column for column in key_metric_columns if column in recipe_overview.columns]
    key_metric_frame = recipe_overview[available_columns].copy()
    display(key_metric_frame)
    display(recipe_overview)
    display(Markdown("上表是集中指标看板；`recipe_overview` 则保留完整细项。`requested_feature_count` 对应配置请求特征数，`used_feature_count` 对应实际进入训练的特征数。"))


,recipe,rolling_rank_ic_ir,rolling_topk_mean_excess_return_4w,rolling_eval_date_count,rolling_eval_date_start,rolling_eval_date_end,rolling_net_total_return,rolling_max_drawdown,walk_forward_rank_ic_ir,walk_forward_topk_mean_excess_return_4w,walk_forward_eval_date_count,walk_forward_eval_date_start,walk_forward_eval_date_end,walk_forward_net_total_return,walk_forward_max_drawdown,latest_score_dispersion,latest_top10_unique_score_count,latest_actual_hold_count,latest_blocked_sell_count
0,baseline,0.122610,-0.005840,52,2025-02-07,2026-01-30,0.053016,-0.041319,0.240718,0.000242,200,2022-03-11,2026-01-30,0.651681,-0.205025,0.079121,None,10.0,0.0
1,binary_4w,-0.018405,0.025284,52,2025-03-07,2026-03-06,0.557754,-0.150152,-0.295961,-0.002677,204,2022-03-11,2026-03-06,0.196119,-0.367342,0.005716,None,12.0,0.0
2,huber_8w,0.152383,-0.006774,52,2025-02-07,2026-01-30,0.122716,-0.061251,0.259381,0.000286,200,2022-03-11,2026-01-30,0.678031,-0.148861,0.102442,None,10.0,0.0
3,mae_4w,0.024579,-0.004176,52,2025-03-07,2026-03-06,0.123629,-0.048981,0.149865,-0.002897,204,2022-03-11,2026-03-06,0.308333,-0.281078,0.063948,None,12.0,0.0
4,rank_blended,-0.040001,-0.001873,52,2025-02-07,2026-01-30,0.051091,-0.083464,0.229003,0.000041,200,2022-03-11,2026-01-30,0.396291,-0.165301,0.621727,None,11.0,0.0


,recipe,path,requested_feature_count,used_feature_count,rolling_rank_ic_ir,rolling_topk_mean_excess_return_4w,rolling_eval_date_count,rolling_eval_date_start,rolling_eval_date_end,rolling_report_date_start,...,walk_forward_excess_total_return,walk_forward_max_drawdown,walk_forward_excess_drawdown,walk_forward_cost_drag,walk_forward_turnover_mean,latest_score_dispersion,latest_top10_unique_score_count,latest_actual_hold_count,latest_blocked_sell_count,latest_score_rows
0,baseline,/Volumes/Repository/Projects/TradingNexus/Valu...,None,63,0.122610,-0.005840,52,2025-02-07,2026-01-30,2025-02-14,...,0.407493,-0.205025,-0.171443,0.066667,0.658846,0.079121,None,10.0,0.0,300
1,binary_4w,/Volumes/Repository/Projects/TradingNexus/Valu...,None,63,-0.018405,0.025284,52,2025-03-07,2026-03-06,2025-03-14,...,-0.049956,-0.367342,-0.364946,0.089619,0.877546,0.005716,None,12.0,0.0,300
2,huber_8w,/Volumes/Repository/Projects/TradingNexus/Valu...,None,63,0.152383,-0.006774,52,2025-02-07,2026-01-30,2025-02-14,...,0.433843,-0.148861,-0.155633,0.065030,0.638476,0.102442,None,10.0,0.0,300
3,mae_4w,/Volumes/Repository/Projects/TradingNexus/Valu...,None,63,0.024579,-0.004176,52,2025-03-07,2026-03-06,2025-03-14,...,0.062257,-0.281078,-0.179557,0.096103,0.937798,0.063948,None,12.0,0.0,300
4,rank_blended,/Volumes/Repository/Projects/TradingNexus/Valu...,None,63,-0.040001,-0.001873,52,2025-02-07,2026-01-30,2025-02-14,...,0.152103,-0.165301,-0.157661,0.138777,1.403333,0.621727,None,11.0,0.0,300


上表是集中指标看板；`recipe_overview` 则保留完整细项。`requested_feature_count` 对应配置请求特征数，`used_feature_count` 对应实际进入训练的特征数。

## 6. 研究价值看板

In [5]:
available_recipes = artifact_view["recipe_names"]
focus_recipe = FOCUS_RECIPE if FOCUS_RECIPE in available_recipes else (available_recipes[0] if available_recipes else None)

if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    feature_prefilter = recipe_data["feature_prefilter"]
    signal_diagnostics = recipe_data["signal_diagnostics"].copy()
    portfolio_diagnostics = recipe_data["portfolio_diagnostics"].copy()
    slice_regime_summary = recipe_data["slice_regime_summary"].copy()

    research_value_board = pd.DataFrame(
        [
            {
                "recipe": focus_recipe,
                "requested_feature_count": _safe_max_numeric(feature_prefilter, "requested_feature_count"),
                "used_feature_count": _safe_max_numeric(feature_prefilter, "selected_feature_count"),
                "latest_score_dispersion": _safe_last_numeric(signal_diagnostics, "score_dispersion"),
                "latest_top10_unique_score_count": _safe_last_numeric(signal_diagnostics, "topk_unique_score_count"),
                "latest_top10_unique_score_ratio": _safe_last_numeric(signal_diagnostics, "topk_unique_score_ratio"),
                "latest_actual_hold_count": _safe_last_numeric(portfolio_diagnostics, "actual_hold_count"),
                "latest_blocked_sell_count": _safe_last_numeric(portfolio_diagnostics, "blocked_sell_count"),
                "slice_row_count": int(len(slice_regime_summary)),
            }
        ]
    )
    display(research_value_board)

    if not signal_diagnostics.empty and "feature_date" in signal_diagnostics.columns:
        signal_diagnostics["feature_date"] = pd.to_datetime(signal_diagnostics["feature_date"])
        _show_line(
            signal_diagnostics,
            x="feature_date",
            y=["score_dispersion", "topk_unique_score_ratio"],
            title=f"{focus_recipe}: 分数离散度与 TopK 唯一分数比例",
        )


,recipe,requested_feature_count,used_feature_count,latest_score_dispersion,latest_top10_unique_score_count,latest_top10_unique_score_ratio,latest_actual_hold_count,latest_blocked_sell_count,slice_row_count
0,huber_8w,None,None,0.102442,None,1.0,10.0,0.0,69


## 7. 执行与风险诊断

In [6]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    portfolio_diagnostics = recipe_data["portfolio_diagnostics"].copy()
    execution_diff_summary = recipe_data["execution_diff_summary"].copy()
    rolling_native_report = recipe_data["rolling_native_report"].copy()
    walk_forward_native_report = recipe_data["walk_forward_native_report"].copy()
    rolling_monthly_heatmap = recipe_data["rolling_native_monthly_return_heatmap"].copy()
    rolling_annual_heatmap = recipe_data["rolling_native_annual_return_heatmap"].copy()
    walk_forward_monthly_heatmap = recipe_data["walk_forward_native_monthly_return_heatmap"].copy()
    walk_forward_annual_heatmap = recipe_data["walk_forward_native_annual_return_heatmap"].copy()

    display(portfolio_diagnostics.tail(12) if not portfolio_diagnostics.empty else pd.DataFrame())
    display(execution_diff_summary if not execution_diff_summary.empty else pd.DataFrame())

    if not portfolio_diagnostics.empty and "trade_date" in portfolio_diagnostics.columns:
        portfolio_diagnostics["trade_date"] = pd.to_datetime(portfolio_diagnostics["trade_date"])
        _show_line(
            portfolio_diagnostics,
            x="trade_date",
            y=["target_hold_count", "actual_hold_count", "blocked_sell_count"],
            title=f"{focus_recipe}: 目标持仓、实际持仓与 blocked sell",
        )

    if not rolling_native_report.empty and "datetime" in rolling_native_report.columns:
        rolling_native_report["datetime"] = pd.to_datetime(rolling_native_report["datetime"])
        _show_line(
            rolling_native_report,
            x="datetime",
            y=["net_value", "benchmark_value"],
            title=f"{focus_recipe}: rolling 净值 vs benchmark",
            height=420,
        )

    if not walk_forward_native_report.empty and "datetime" in walk_forward_native_report.columns:
        walk_forward_native_report["datetime"] = pd.to_datetime(walk_forward_native_report["datetime"])
        _show_line(
            walk_forward_native_report,
            x="datetime",
            y=["net_value", "benchmark_value"],
            title=f"{focus_recipe}: walk-forward 净值 vs benchmark",
            height=420,
        )

    _render_return_heatmap(
        rolling_monthly_heatmap,
        title=f"{focus_recipe}: rolling 月度净收益热力图",
        x_label="Month",
        y_label="Year",
    )
    _render_return_heatmap(
        rolling_annual_heatmap,
        title=f"{focus_recipe}: rolling 年度净收益热力图",
        x_label="Year",
        y_label="Metric",
    )
    _render_return_heatmap(
        walk_forward_monthly_heatmap,
        title=f"{focus_recipe}: walk-forward 月度净收益热力图",
        x_label="Month",
        y_label="Year",
    )
    _render_return_heatmap(
        walk_forward_annual_heatmap,
        title=f"{focus_recipe}: walk-forward 年度净收益热力图",
        x_label="Year",
        y_label="Metric",
    )


,signal_date,trade_date,target_hold_count,actual_hold_count,blocked_sell_count,blocked_sell_codes,residual_hold_count,rebalance_interval_steps,hold_buffer_rank,coverage,score_dispersion,score_unique_count,topk_unique_score_ratio,topk_overlap_prev,future_return_top_bottom_decile_spread,excess_return_top_bottom_decile_spread,recipe,bundle
238,2025-11-07,2025-11-14,10,10,0,NaN,0,1,15,300.0,0.076885,299.0,1.0,0.7,0.021748,0.021748,huber_8w,walk_forward
239,2025-11-14,2025-11-21,10,11,0,NaN,1,1,15,300.0,0.058775,299.0,1.0,0.7,-0.007074,-0.007074,huber_8w,walk_forward
240,2025-11-21,2025-11-28,10,11,0,NaN,2,1,15,300.0,0.057716,294.0,1.0,0.7,-0.002340,-0.002340,huber_8w,walk_forward
241,2025-11-28,2025-12-05,10,10,0,NaN,0,1,15,298.0,0.053648,297.0,1.0,0.7,-0.048602,-0.048602,huber_8w,walk_forward
242,2025-12-05,2025-12-12,10,10,0,NaN,0,1,15,298.0,0.053838,297.0,1.0,0.6,-0.036798,-0.036798,huber_8w,walk_forward
243,2025-12-12,2025-12-19,10,10,0,NaN,0,1,15,298.0,0.040184,280.0,0.9,0.7,-0.084267,-0.084267,huber_8w,walk_forward
244,2025-12-19,2025-12-26,10,10,0,NaN,0,1,15,300.0,0.047769,291.0,1.0,0.7,-0.128672,-0.128672,huber_8w,walk_forward
245,2025-12-26,2026-01-02,10,10,0,NaN,0,1,15,299.0,0.046967,275.0,0.9,0.7,-0.110645,-0.110645,huber_8w,walk_forward
246,2026-01-02,2026-01-09,10,10,0,NaN,0,1,15,299.0,0.046535,279.0,0.9,0.7,-0.090554,-0.090554,huber_8w,walk_forward
247,2026-01-09,2026-01-16,10,10,0,NaN,0,1,15,299.0,0.047434,280.0,0.8,0.5,0.027554,0.027554,huber_8w,walk_forward


,recipe,bundle,native_final_net_value,validation_final_net_value,native_minus_validation_return,native_max_drawdown,validation_max_drawdown
0,huber_8w,rolling,1.122716e+06,1.131751e+06,-0.009035,-0.061251,-0.055167
1,huber_8w,walk_forward,1.678031e+06,1.517916e+06,0.160114,-0.148861,-0.192019


## 8. 切片稳定性与特征重要性

In [7]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    slice_regime_summary = recipe_data["slice_regime_summary"].copy()
    rolling_feature_importance = recipe_data["rolling_feature_importance"].copy()

    if not slice_regime_summary.empty:
        display(slice_regime_summary.sort_values(["bundle", "slice_type", "coverage"], ascending=[True, True, False]).head(30))
    else:
        display(pd.DataFrame())

    if not rolling_feature_importance.empty:
        top_features = (
            rolling_feature_importance.groupby("feature", as_index=False)["importance_gain"]
            .mean()
            .sort_values("importance_gain", ascending=False)
            .head(20)
        )
        display(top_features)
        fig = px.bar(
            top_features.sort_values("importance_gain", ascending=True),
            x="importance_gain",
            y="feature",
            orientation="h",
            title=f"{focus_recipe}: rolling 平均特征重要性（gain）",
        )
        fig.update_layout(template="plotly_white", height=520)
        fig.show()


,recipe,bundle,slice_type,slice_value,coverage,score_dispersion,mean_future_return_4w,mean_excess_return_4w
0,huber_8w,rolling,feature_year,2025,14040,0.085727,0.017875,-0.000973
1,huber_8w,rolling,feature_year,2026,1498,0.069148,0.007369,-0.007790
24,huber_8w,rolling,l1_name,电子,1603,0.095624,0.030678,0.012148
33,huber_8w,rolling,l1_name,非银金融,1351,0.056272,0.008278,-0.010114
32,huber_8w,rolling,l1_name,银行,1206,0.067128,0.005273,-0.013548
23,huber_8w,rolling,l1_name,电力设备,1187,0.060006,0.033461,0.015427
35,huber_8w,rolling,l1_name,NaN,1087,0.071899,0.016293,-0.002210
11,huber_8w,rolling,l1_name,医药生物,1061,0.082880,-0.001892,-0.020190
7,huber_8w,rolling,l1_name,交通运输,782,0.064372,0.012711,-0.005942
28,huber_8w,rolling,l1_name,计算机,703,0.087005,-0.001123,-0.020054


,feature,importance_gain
12,downside_volatility_8w,1797.283618
30,ma50,1531.239414
9,dea,1477.979381
31,macd_hist,1475.275204
4,atr,1394.800050
56,roa_ttm_delta_8w,874.366092
39,ni_cfo_corr_3y,790.080650
29,ma20,669.401817
10,debt_ratio,557.689496
60,volatility_8w,535.584425


## 9. 最新一期信号快照

In [8]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    latest_score_frame = artifact_view["recipes"][focus_recipe]["latest_score_frame"].copy()
    if latest_score_frame.empty:
        display(Markdown("当前 recipe 没有最新一期快照。"))
    else:
        display(latest_score_frame.head(20))
        if {"l1_name", "score"}.issubset(latest_score_frame.columns):
            industry_snapshot = (
                latest_score_frame.head(50)
                .groupby("l1_name", dropna=False)
                .agg(stock_count=("instrument", "count"), mean_score=("score", "mean"))
                .reset_index()
                .sort_values(["mean_score", "stock_count"], ascending=[False, False])
            )
            display(industry_snapshot)

display(Markdown("如果要发布线上 snapshot，请把 `CONFIG['publish_model']` 改成 `True`，然后重新运行 workflow cell。"))


,datetime,instrument,score,open,close,volume,amount,future_return_4w,future_return_8w,label_excess_return_4w,label_excess_return_8w,model_label_raw,model_label,in_csi300,l1_name,l2_name,l3_name,macro_phase,macro_industry_match,feature_date
0,2026-04-03,601009.SH,0.223354,11.15,11.34,1506589,1.707415e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-03
1,2026-04-03,601229.SH,0.216474,9.76,9.89,1177767,1.169341e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-03
2,2026-04-03,601018.SH,0.192955,4.08,4.06,1362093,5.580658e+05,NaN,NaN,NaN,NaN,NaN,NaN,True,交通运输,航运港口,港口,REFLATION,0,2026-04-03
3,2026-04-03,600919.SH,0.187086,10.75,10.82,2689329,2.929023e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-03
4,2026-04-03,600362.SH,0.186743,43.70,44.55,873084,3.834305e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,有色金属,工业金属,铜,REFLATION,0,2026-04-03
5,2026-04-03,601377.SH,0.172831,5.99,5.94,2560214,1.517063e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,非银金融,证券Ⅱ,证券Ⅲ,REFLATION,1,2026-04-03
6,2026-04-03,601939.SH,0.171330,9.33,9.45,3435363,3.277288e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,国有大型银行Ⅱ,国有大型银行Ⅲ,REFLATION,0,2026-04-03
7,2026-04-03,601077.SH,0.170905,7.02,7.06,1196935,8.427847e+05,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,农商行Ⅱ,农商行Ⅲ,REFLATION,0,2026-04-03
8,2026-04-03,601988.SH,0.162449,5.55,5.87,12850762,7.466148e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,国有大型银行Ⅱ,国有大型银行Ⅲ,REFLATION,0,2026-04-03
9,2026-04-03,603799.SH,0.154384,59.00,60.30,1633719,9.707969e+06,NaN,NaN,NaN,NaN,NaN,NaN,True,有色金属,能源金属,钴,REFLATION,0,2026-04-03


,l1_name,stock_count,mean_score
14,银行,13,0.142368
5,基础化工,1,0.119117
8,有色金属,5,0.108506
1,公用事业,5,0.100628
11,电子,4,0.100024
15,非银金融,5,0.092525
3,商贸零售,1,0.089050
0,交通运输,4,0.081289
6,家用电器,1,0.071917
9,机械设备,2,0.070986


如果要发布线上 snapshot，请把 `CONFIG['publish_model']` 改成 `True`，然后重新运行 workflow cell。

## 10. 后续动作

建议优先按这个顺序继续推进：
- 先比较 `baseline` 与 candidate 的 `execution_diff_summary`，确认收益问题到底来自信号还是执行。
- 再看 `research_value_board` 与 `slice_regime_summary`，确认分数粒度、年份稳定性、行业集中度是否改善。
- 最后才决定是否发布 snapshot，避免把“会排不会赚”的配方直接推向下游。